# Food Delivery Operations Analytics — Python EDA

This notebook uses only the `Raw_Data` sheet from the uploaded workbook. It preserves the original raw file and creates an analysis copy with supported derived columns.

In [1]:
from pathlib import Path
import runpy
OUT = Path('/home/ubuntu/food_delivery_eda')
# The implementation below is the complete reproducible Python EDA script.
env = runpy.run_path(str(OUT / 'build_eda.py'))

EDA_COMPLETE
raw_shape (50000, 37)
final_shape (50000, 43)
kpis
              KPI  Calculated value
     Total Orders         50000.000
 Completed Orders         49132.000
 Cancelled Orders           868.000
  Completion Rate            98.264
Cancellation Rate             1.736
validation
                              Metric  Reference value  Calculated value  Matches reference
                        Total Orders        50000.000      50000.000000               True
                    Cancelled Orders          868.000        868.000000               True
                 Cancellation Rate %            1.736          1.736000               True
          Changed Mind cancellations          153.000        153.000000               True
    Customer Cancelled cancellations          154.000        154.000000               True
     Driver Unassigned cancellations          386.000        386.000000               True
       Long Prep Delay cancellations           15.000         15.000000 

## 1. Import dataset

In [2]:
print('Input workbook: food_delivery_operations_intelligence.xlsx')
print('Raw_Data shape:', env['raw_shape'])
print(env['df'].head().to_string(index=False))
print(env['df'].tail().to_string(index=False))

Input workbook: food_delivery_operations_intelligence.xlsx
Raw_Data shape: (50000, 37)
  order_id customer_id restaurant_id driver_id     order_timestamp order_date  order_hour day_of_week  is_weekend   city delivery_area  customer_age customer_type restaurant_type restaurant_primary_category  restaurant_rating  items_count  subtotal  discount_percent  tax_amount  service_fee  delivery_fee  order_total payment_method  tip_amount  distance_km weather traffic_level  delivery_partner_experience_months  delivery_partner_rating  restaurant_preparation_time_minutes  estimated_delivery_time_minutes  actual_delivery_time_minutes  late_delivery order_status cancellation_reason  customer_rating  Cancellation_Flag  Completed_Flag  Delivery_Delay_Minutes  Late_Delivery_Flag Distance_Category     Hour_Category
ORD_000001   CUS_00001      RES_0008  DRV_0075 2025-10-05 15:54:00 2025-10-05          15      Sunday           1 City_C City_C_Area_3            40       Premium   Casual Dining             

## 2. Basic inspection

In [3]:
print('Columns:', list(env['df'].columns[:37]))
print('Source dtypes:\n', env['df'].iloc[:, :37].dtypes.to_string())
print('Numeric source columns:', list(env['df'].select_dtypes(include='number').columns))
print('Categorical/object source columns:', list(env['df'].select_dtypes(include=['object','category']).columns))

Columns: ['order_id', 'customer_id', 'restaurant_id', 'driver_id', 'order_timestamp', 'order_date', 'order_hour', 'day_of_week', 'is_weekend', 'city', 'delivery_area', 'customer_age', 'customer_type', 'restaurant_type', 'restaurant_primary_category', 'restaurant_rating', 'items_count', 'subtotal', 'discount_percent', 'tax_amount', 'service_fee', 'delivery_fee', 'order_total', 'payment_method', 'tip_amount', 'distance_km', 'weather', 'traffic_level', 'delivery_partner_experience_months', 'delivery_partner_rating', 'restaurant_preparation_time_minutes', 'estimated_delivery_time_minutes', 'actual_delivery_time_minutes', 'late_delivery', 'order_status', 'cancellation_reason', 'customer_rating']
Source dtypes:
 order_id                                          str
customer_id                                       str
restaurant_id                                     str
driver_id                                         str
order_timestamp                        datetime64[us]
order_date    

/tmp/ipykernel_2465/4058793317.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print('Categorical/object source columns:', list(env['df'].select_dtypes(include=['object','category']).columns))


## 3. Data quality check

In [4]:
print('Total missing cells:', env['raw_nulls'])
print('Duplicate full rows:', env['raw_dups'])
print('Duplicate order IDs:', env['raw_dup_ids'])
print(env['null_summary'][env['null_summary']['null_count']>0].to_string(index=False))

Total missing cells: 52604
Duplicate full rows: 0
Duplicate order IDs: 0
                      column  null_count  null_pct
         cancellation_reason       49132    98.264
      Delivery_Delay_Minutes         868     1.736
actual_delivery_time_minutes         868     1.736
             customer_rating         868     1.736
               late_delivery         868     1.736
                  tip_amount         868     1.736


## 4. Data cleaning

In [5]:
print('Before shape:', env['raw_shape'])
print('After shape:', env['df'].shape)
print('Cleaning decision: parse dates; retain status-aligned missing values; remove no rows.')

Before shape: (50000, 37)
After shape: (50000, 43)
Cleaning decision: parse dates; retain status-aligned missing values; remove no rows.


## 5. Feature engineering

In [6]:
print(env['df'][['Cancellation_Flag','Completed_Flag','Delivery_Delay_Minutes','Late_Delivery_Flag','Distance_Category','Hour_Category']].head().to_string(index=False))
print('Derived columns:', ['Cancellation_Flag','Completed_Flag','Delivery_Delay_Minutes','Late_Delivery_Flag','Distance_Category','Hour_Category'])

 Cancellation_Flag  Completed_Flag  Delivery_Delay_Minutes  Late_Delivery_Flag Distance_Category     Hour_Category
                 0               1                     4.0                   0            0–2 km Afternoon (12–16)
                 0               1                     4.0                   0           >2–5 km   Evening (17–20)
                 0               1                     9.0                   1          >8–12 km   Evening (17–20)
                 0               1                     8.0                   1           >5–8 km Afternoon (12–16)
                 0               1                    10.0                   1           >5–8 km   Evening (17–20)
Derived columns: ['Cancellation_Flag', 'Completed_Flag', 'Delivery_Delay_Minutes', 'Late_Delivery_Flag', 'Distance_Category', 'Hour_Category']


## 6. KPI analysis and reference validation

In [7]:
print(env['kpi'].to_string(index=False))
print('\nReference validation:')
print(env['validation'].to_string(index=False))

              KPI  Calculated value
     Total Orders         50000.000
 Completed Orders         49132.000
 Cancelled Orders           868.000
  Completion Rate            98.264
Cancellation Rate             1.736

Reference validation:
                              Metric  Reference value  Calculated value  Matches reference
                        Total Orders        50000.000      50000.000000               True
                    Cancelled Orders          868.000        868.000000               True
                 Cancellation Rate %            1.736          1.736000               True
          Changed Mind cancellations          153.000        153.000000               True
    Customer Cancelled cancellations          154.000        154.000000               True
     Driver Unassigned cancellations          386.000        386.000000               True
       Long Prep Delay cancellations           15.000         15.000000               True
Restaurant Unavailable cancellati

## 7. Cancellation analysis

In [8]:
print('Cancellation reasons:\n', env['cancel_reasons'].to_string(index=False))
print('\nCity cancellation rates:\n', env['city_cancel'].to_string(index=False))
print('\nHour cancellations:\n', env['hour'].to_string(index=False))
print('\nDay cancellations:\n', env['day_order'].to_string(index=False))

Cancellation reasons:
    Cancellation Reason  Cancelled Orders  Share of Cancellations %
     Driver Unassigned               386                 44.470046
Restaurant Unavailable               160                 18.433180
    Customer Cancelled               154                 17.741935
          Changed Mind               153                 17.626728
       Long Prep Delay                15                  1.728111

City cancellation rates:
   city  Orders  Cancelled_Orders  Cancellation_Rate %
City_A   19888               361             1.815165
City_B   14885               258             1.733289
City_C   10178               172             1.689919
City_D    5049                77             1.525054

Hour cancellations:
  order_hour  Orders  Cancellations
          0     429              6
          1     437              7
          2     408              3
          3     407              3
          4     425              3
          5     404              9
          6

## 8. Delivery performance

In [9]:
print(env['delivery_kpis'].to_string(index=False))

                                   Metric        Value
       Average preparation time (minutes)    15.543495
Average estimated delivery time (minutes)    30.919543
   Average actual delivery time (minutes)    35.152365
         Average delivery delay (minutes)     4.232822
                          Late deliveries 17968.000000
                       Late delivery rate    36.570870


## 9. City analysis

In [10]:
print(env['city'].to_string(index=False))

  city  Orders  Completed_Orders  Cancelled_Orders  Avg_Delivery_Time  Late_Delivery_Rate  Avg_Order_Value  Cancellation_Rate
City_A   19888             19527               361          34.991294           37.594100        36.769089           1.815165
City_B   14885             14627               258          36.008546           38.189649        41.093784           1.733289
City_C   10178             10006               172          34.056666           33.120128        34.131735           1.689919
City_D    5049              4972                77          35.471239           34.734513        43.204124           1.525054


## 10. Traffic analysis

In [11]:
print(env['traffic'].to_string(index=False))

traffic_level  Orders  Completed_Orders  Cancelled_Orders  Avg_Delivery_Time  Late_Delivery_Rate  Avg_Order_Value  Cancellation_Rate
         High   15313             14948               365          39.888079           38.654001        37.852119           2.383596
          Low   15165             14987               178          30.593047           34.663375        38.321997           1.173755
       Medium   19522             19197               325          35.024275           36.437985        38.299993           1.664788


## 11. Weather analysis

In [12]:
print(env['weather'].to_string(index=False))

weather  Orders  Completed_Orders  Cancelled_Orders  Avg_Delivery_Time  Late_Delivery_Rate  Avg_Order_Value  Cancellation_Rate
  Clear   28248             27857               391          33.555803           24.492946        38.303042           1.384169
 Cloudy   12810             12581               229          35.121532           38.764804        37.297985           1.787666
   Rain    7627              7427               200          39.159418           69.180019        38.606579           2.622263
  Storm    1315              1267                48          47.072612           89.187056        41.255627           3.650190


## 12. Distance analysis

In [13]:
print(env['df']['distance_km'].describe().to_string())
print('\nDistance category summary:\n', env['distance'].to_string(index=False))

count    50000.000000
mean         4.730632
std          2.780190
min          0.800000
25%          2.800000
50%          4.100000
75%          5.900000
max         20.000000

Distance category summary:
 Distance_Category  Orders  Completed_Orders  Cancelled_Orders  Avg_Delivery_Time  Late_Delivery_Rate  Avg_Order_Value  Cancellation_Rate
           0–2 km    5333              5267                66          22.594266           29.428517        37.010392           1.237577
          >2–5 km   27301             26818               483          30.072339           34.521590        37.697650           1.769166
          >5–8 km   11917             11711               206          41.196140           40.227137        38.266439           1.728623
         >8–12 km    4195              4106                89          54.871651           44.861179        40.925282           2.121573
           >12 km    1254              1230                24          76.317886           49.349593        43

## 13. Restaurant analysis

In [14]:
print('Restaurant type:\n', env['restaurant_type'].to_string(index=False))
print('\nCuisine/category:\n', env['cuisine'].to_string(index=False))
print('\nRestaurant rating summary:\n', env['restaurant_rating'].to_string(index=False))

Restaurant type:
 restaurant_type  Orders  Completed_Orders  Cancelled_Orders  Avg_Delivery_Time  Late_Delivery_Rate  Avg_Order_Value  Cancellation_Rate
           Cafe    8140              7999               141          31.290786           35.254407        22.140689           1.732187
  Casual Dining   13063             12848               215          38.599315           37.064134        52.321824           1.645870
      Fast Food   27593             27106               487          34.110418           36.884823        33.172616           1.764940
    Fine Dining    1204              1179                25          47.743851           32.909245       107.506537           2.076412

Cuisine/category:
 restaurant_primary_category  Orders  Completed_Orders  Cancelled_Orders  Avg_Delivery_Time  Late_Delivery_Rate  Avg_Order_Value  Cancellation_Rate
                      Asian    3799              3744                55          36.653846           37.259615        46.415841           1.

## 14. Customer and payment analysis

In [15]:
print('Customer type:\n', env['customer_type'].to_string(index=False))
print('\nPayment method:\n', env['payment'].to_string(index=False))

Customer type:
 customer_type  Orders  Completed_Orders  Cancelled_Orders  Avg_Delivery_Time  Late_Delivery_Rate  Avg_Order_Value  Cancellation_Rate
          New   10175             10008               167          35.727718           37.559952        40.679512           1.641278
      Premium   14727             14452               275          34.710559           34.922502        33.965364           1.867319
      Regular   25098             24672               426          35.177772           37.135214        39.618819           1.697346

Payment method:
 payment_method  Orders  Completed_Orders  Cancelled_Orders  Avg_Delivery_Time  Late_Delivery_Rate  Avg_Order_Value  Cancellation_Rate
          Cash   15050             14811               239          35.199176           37.019783        38.591649           1.588040
   Credit Card   24971             24523               448          35.179709           36.577906        38.058980           1.794081
Digital Wallet    9979          

## 15. Visualization

In [16]:
print('Charts created:')
for c in env['charts']:
    print('-', c)

Charts created:
- 01_completed_vs_cancelled.png
- 02_cancellation_reasons.png
- 03_cancellation_rate_city.png
- 04_cancellation_reasons_city.png
- 05_orders_cancellations_hour.png
- 06_orders_cancellations_day.png
- 07_delivery_time_distribution.png
- 08_late_vs_ontime.png
- 09_cancellation_rate_traffic.png
- 10_cancellation_rate_weather.png
- 11_cancellation_rate_distance.png
- 12_delivery_time_traffic.png


## 16. Business insights

In [17]:
df=env['df']; completed_df=env['completed_df']; cancel_reasons=env['cancel_reasons']; city_cancel=env['city_cancel']; traffic=env['traffic']; weather=env['weather']; distance=env['distance']
lead=cancel_reasons.iloc[0]
print(f"1. {lead['Cancellation Reason']} is the largest cancellation reason: {int(lead['Cancelled Orders'])} orders ({lead['Share of Cancellations %']:.2f}% of cancellations).")
print(f"2. Late deliveries: {int(completed_df['Late_Delivery_Flag'].sum())} of {len(completed_df)} completed orders ({completed_df['Late_Delivery_Flag'].mean()*100:.2f}%).")
print(f"3. City cancellation-rate range: {city_cancel['Cancellation_Rate %'].min():.2f}% to {city_cancel['Cancellation_Rate %'].max():.2f}%.")
print('These are confirmed associations, not causal claims.')

1. Driver Unassigned is the largest cancellation reason: 386 orders (44.47% of cancellations).
2. Late deliveries: 17968 of 49132 completed orders (36.57%).
3. City cancellation-rate range: 1.53% to 1.82%.
These are confirmed associations, not causal claims.


## 17. Final summary

In [18]:
print('Final cleaned DataFrame shape:', env['df'].shape)
print('Report:', OUT / 'food_delivery_eda_report.md')
print('Cleaned CSV:', OUT / 'food_delivery_eda_cleaned.csv')
print('Charts:', len(env['charts']))

Final cleaned DataFrame shape: (50000, 43)
Report: /home/ubuntu/food_delivery_eda/food_delivery_eda_report.md
Cleaned CSV: /home/ubuntu/food_delivery_eda/food_delivery_eda_cleaned.csv
Charts: 12


## 18. Interview questions and simple answers

In [19]:
questions = [
('What was the dataset size?', f"{len(env['df']):,} orders and 37 source columns."),
('What was the cancellation rate?', f"{env['cancellation_rate']:.3f}%."),
('How did you handle missing values?', 'I retained status-aligned missingness and did not blindly impute.'),
('How did you check duplicates?', 'Full-row and order-ID duplicate counts were both zero.'),
('Did you prove causation?', 'No; EDA reports observed differences and follow-up questions.'),
]
for q,a in questions: print('Q:',q,'\nA:',a,'\n')

Q: What was the dataset size? 
A: 50,000 orders and 37 source columns. 

Q: What was the cancellation rate? 
A: 1.736%. 

Q: How did you handle missing values? 
A: I retained status-aligned missingness and did not blindly impute. 

Q: How did you check duplicates? 
A: Full-row and order-ID duplicate counts were both zero. 

Q: Did you prove causation? 
A: No; EDA reports observed differences and follow-up questions. 

